In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/psaimanikanta/hackcrew-ds/data_C_Ours_test.csv
/kaggle/input/datasets/psaimanikanta/hackcrew-ds/data_C_Ours_train.csv


In [2]:
import numpy as np
import pandas as pd
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_recall_curve, classification_report,
                             confusion_matrix, fbeta_score)

RANDOM_STATE = 42

# Data Loading and EDA

In [3]:
train = pd.read_csv('/kaggle/input/datasets/psaimanikanta/hackcrew-ds/data_C_Ours_train.csv')
test = pd.read_csv('/kaggle/input/datasets/psaimanikanta/hackcrew-ds/data_C_Ours_test.csv')

print("train:", train.shape, "| test:", test.shape)
print(train.target.value_counts(), "\n")
train.head(2)

train: (10826, 8) | test: (2706, 8)
target
1    5413
0    5413
Name: count, dtype: int64 



,Unnamed: 0,cwe_id,code,repo,path,url,sha,target
0,3530,NVD-CWE-Other,"static int ras_getdatastd(jas_stream_t *in, ra...",visit repo url,src/libjasper/ras/ras_dec.c,https://github.com/mdadams/jasper,106829516887531,1
1,4682,CWE-732,char *M_fs_path_tmpdir(M_fs_system_t sys_type)...,visit repo url,base/fs/m_fs_path.c,https://github.com/Monetra/mstdlib,88247877163772,1


In [4]:
train[train["target"] == 0].head(5)

,Unnamed: 0,cwe_id,code,repo,path,url,sha,target
2,4537,['CWE-20'],"static int make_indexed_dir(handle_t *handle, ...",linux-2.6,NaN,NaN,287883023626102361284836105016297785515,0
4,4426,['CWE-264'],"ssize_t sock_no_sendpage(struct socket *sock, ...",linux-2.6,NaN,NaN,78434298454054858077706649493150329771,0
5,4842,['CWE-189'],"write_ecryptfs_flags(char *page_virt, struct e...",linux-2.6,NaN,NaN,298313984179014176993334665379279089654,0
7,5433,['CWE-476'],static int is_efer_nx(void)\n{\n\tunsigned lon...,linux-2.6,NaN,NaN,260499358369119256138692990202697785976,0
8,3439,['CWE-264'],"static long do_tee(struct file *in, struct fil...",linux-2.6,NaN,NaN,235949820129859338460165285264020714539,0


In [5]:
train[train["target"] == 1].head(5)

,Unnamed: 0,cwe_id,code,repo,path,url,sha,target
0,3530,NVD-CWE-Other,"static int ras_getdatastd(jas_stream_t *in, ra...",visit repo url,src/libjasper/ras/ras_dec.c,https://github.com/mdadams/jasper,106829516887531,1
1,4682,CWE-732,char *M_fs_path_tmpdir(M_fs_system_t sys_type)...,visit repo url,base/fs/m_fs_path.c,https://github.com/Monetra/mstdlib,88247877163772,1
3,2148,CWE-476,struct btrfs_device *btrfs_find_device(struct ...,visit repo url,fs/btrfs/volumes.c,https://github.com/torvalds/linux,239983200436702,1
6,2181,CWE-416,"static int gup_pte_range(pmd_t pmd, unsigned l...",visit repo url,mm/gup.c,https://github.com/torvalds/linux,32482761570393,1
9,5527,CWE-125,ast2obj_mod(void* _o)\n{\n mod_ty o = (mod_...,visit repo url,ast3/Python/Python-ast.c,https://github.com/python/typed_ast,192939688364872,1


In [6]:
url_check = train.groupby("url")["target"].nunique()

print("URLs mapping to only one target:")
print((url_check == 1).sum())

print("Total unique URLs:")
print(len(url_check))

URLs mapping to only one target:
402
Total unique URLs:
402


In [7]:
print("no. of repeating Sha:",10826- train["sha"].nunique())

no. of repeating Sha: 851


In [8]:
print(pd.crosstab(train["path"].isnull(), train["target"]))
print()

print(pd.crosstab(
    train["repo"] == "visit repo url",
    train["target"]
))
print()

print(pd.crosstab(
    train["cwe_id"].astype(str).str.startswith("["),
    train["target"]
))
print()

print(pd.crosstab(
    train["url"].isnull(),
    train["target"]
))

target     0     1
path              
False      0  5413
True    5413     0

target     0     1
repo              
False   5413     0
True       0  5413

target     0     1
cwe_id            
False      0  5413
True    5413     0

target     0     1
url               
False      0  5413
True    5413     0


1. We found that other than code colunm all others are misleading
2. We will take only code colunm and we will train tha model

In [9]:
print("duplicate code rows in train:", train.code.duplicated().sum())
print("codes with conflicting labels:",
      (train.groupby("code").target.nunique() > 1).sum())

# Remove duplicate code rows in train
train = train.drop_duplicates("code").reset_index(drop=True)


# Remove train-test overlap
train_codes = set(train["code"])

# Keep only test samples NOT present in train
test = test[~test["code"].isin(train_codes)].reset_index(drop=True)

# Check overlap again
overlap = test.code.isin(set(train.code)).sum()
print("test rows whose code also appears in train:", overlap)

print("New test shape:", test.shape)

X_train, y_train = train["code"], train["target"].values
X_test, y_test = test["code"], test["target"].values

duplicate code rows in train: 863
codes with conflicting labels: 0
test rows whose code also appears in train: 0
New test shape: (2377, 8)


In [10]:
C_TOKEN = (r"[A-Za-z_][A-Za-z0-9_]*|->|==|!=|<=|>="
           r"|\+\+|--|&&|\|\||<<|>>|[{}()\[\];,*&<>=+\-/%!]")

def build_pipeline():
    word = TfidfVectorizer(lowercase=False, token_pattern=C_TOKEN,
                           ngram_range=(1, 2), min_df=3,
                           max_features=200_000, sublinear_tf=True)
    char = TfidfVectorizer(lowercase=False, analyzer="char_wb",
                           ngram_range=(3, 5), min_df=3,
                           max_features=200_000, sublinear_tf=True)
    return Pipeline([
        ("features", FeatureUnion([("word", word), ("char", char)])),
        ("clf", LogisticRegression(C=2.0, max_iter=3000)),
    ])

pipe = build_pipeline()

In [11]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof = cross_val_predict(pipe, X_train, y_train, cv=cv,
                        method="predict_proba", n_jobs=-1)[:, 1]

print(f"CV ROC-AUC: {roc_auc_score(y_train, oof):.4f}")
print(f"CV PR-AUC : {average_precision_score(y_train, oof):.4f}")

prec, rec, th = precision_recall_curve(y_train, oof)
f2 = (5 * prec * rec) / (4 * prec + rec + 1e-12)
best = np.argmax(f2[:-1])
THRESHOLD = float(th[best])
print(f"\nChosen threshold = {THRESHOLD:.3f} "
      f"(OOF F2={f2[best]:.3f}, precision={prec[best]:.3f}, recall={rec[best]:.3f})")


CV ROC-AUC: 0.9771
CV PR-AUC : 0.9773

Chosen threshold = 0.315 (OOF F2=0.935, precision=0.865, recall=0.954)


In [12]:
pipe.fit(X_train, y_train)
test_proba = pipe.predict_proba(X_test)[:, 1]

print(f"TEST ROC-AUC: {roc_auc_score(y_test, test_proba):.4f}")
print(f"TEST PR-AUC : {average_precision_score(y_test, test_proba):.4f}\n")

test_pred = (test_proba >= THRESHOLD).astype(int)
print(classification_report(y_test, test_pred, digits=3,
                            target_names=["false positive (0)", "real vuln (1)"]))
print("Confusion matrix [rows=truth, cols=pred]:")
print(confusion_matrix(y_test, test_pred))
print(f"\nTEST F2: {fbeta_score(y_test, test_pred, beta=2):.4f}")


TEST ROC-AUC: 0.9832
TEST PR-AUC : 0.9815

                    precision    recall  f1-score   support

false positive (0)      0.970     0.889     0.928      1353
     real vuln (1)      0.868     0.964     0.913      1024

          accuracy                          0.921      2377
         macro avg      0.919     0.927     0.921      2377
      weighted avg      0.926     0.921     0.922      2377

Confusion matrix [rows=truth, cols=pred]:
[[1203  150]
 [  37  987]]

TEST F2: 0.9431


In [13]:
names = pipe.named_steps["features"].get_feature_names_out()
coefs = pipe.named_steps["clf"].coef_[0]
order = np.argsort(coefs)
print("Most VULNERABLE-leaning features:")
print([names[i] for i in order[-20:][::-1]], "\n")
print("Most SAFE-leaning features:")
print([names[i] for i in order[:20]])


Most VULNERABLE-leaning features:
['word__->', 'word__) ;', 'word__true', 'word__; if', 'word__) ,', 'word__; struct', 'word__,', 'word__bool', 'word__* )', 'word__false', 'word__} if', 'word__==', 'word__+', 'word__=', 'word__r', 'word__net', 'word__buf', 'word__) !=', 'word__true ;', 'word__= ('] 

Most SAFE-leaning features:
['word__int', 'char__jas_', 'char__jas', 'word__printk', 'word__printk (', 'char__pfm', 'char__fm_', 'char__pfm_', 'char__sct', 'word__applet', 'char__jpc', 'char__jpc_', 'word__rq', 'word__( KERN_ERR', 'word__current ->', 'word__KERN_ERR', 'word__pRcvBuf', 'char__uct ', 'char__ruct ', 'char__ ja']


In [14]:
joblib.dump({"pipeline": pipe, "threshold": THRESHOLD}, "fp_classifier_model.joblib", compress=3)

_MODEL = pipe

def predict(sample: dict) -> float:
    code_text = sample.get("code", "")
    if not isinstance(code_text, str) or not code_text.strip():
        raise ValueError('sample must contain a non-empty "code" string')
    return float(_MODEL.predict_proba([code_text])[0, 1])

# demo
demo = test.iloc[0]
p = predict({"code": demo.code})
print(f"predicted P(vulnerable) = {p:.3f} | true label = {demo.target} "
      f"| flag for review: {p >= THRESHOLD}")


predicted P(vulnerable) = 0.165 | true label = 0 | flag for review: False
